# Task 3: NLP Preprocessing

Shows what `src/preprocess.py` does to the job descriptions, step by step, with evidence that all four companies end up at the **same quality level** so later comparisons are fair.

Run `python src/preprocess.py` first. Printed text is cut to 300 characters: the full descriptions are copyrighted and this notebook is committed to a public repo.

In [1]:
import sys, pandas as pd
sys.path.insert(0, "../src")
import preprocess as pp

df = pd.read_csv("../data/interim/jobs_clean.csv")
cut = lambda t, n=300: (str(t)[:n] + "...") if len(str(t)) > n else str(t)
df.groupby("company_name").size()

company_name
Anthropic      682
Databricks     926
NVIDIA        2755
OpenAI         818
dtype: int64

## 1. The problem: boilerplate

Every company wraps each posting in the same legal and "about us" text. Left in, it creates fake skill signals. Example: Anthropic's company blurb mentions **GPT-3**, so every Anthropic role would look like it needs GPT-3.

In [2]:
a = df[df.company_name == "Anthropic"]
print("Anthropic postings mentioning GPT-3:  raw", a.job_description.str.contains("GPT-3").sum(),
      " | after cleaning", a.cleaned_description.str.contains("GPT-3").sum(), "of", len(a))

Anthropic postings mentioning GPT-3:  raw 682  | after cleaning 0 of 682


## 2. Method: remove sentences repeated across a company's postings

A sentence that appears in **10% or more** of one company's postings is boilerplate. Digits are replaced first, so "accepted until November 3" and "until November 17" count as the same sentence. 10% was chosen by inspection: everything above it was legal/benefits/about-us text, and real role content (e.g. Databricks' templated architect duties) starts at around 8%. Only one technical sentence sat above 10%: NVIDIA's company blurb saying its GPU "acts as the brains of computers", which should go.

A second pass removes sentences containing phrases that only ever appear in boilerplate (e.g. "equal opportunity", "background check"), which catches reworded versions that each fall under 10%.

In [3]:
for phrase, company in [("public benefit corporation", "Anthropic"), ("Background checks", "OpenAI"),
                        ("Benefits At Databricks", "Databricks"), ("most desirable employers", "NVIDIA")]:
    g = df[df.company_name == company]
    print(f"{company:10} {phrase!r:30} raw {g.job_description.str.contains(phrase).mean():4.0%}"
          f"  ->  clean {g.cleaned_description.str.contains(phrase).mean():4.0%}")

Anthropic  'public benefit corporation'   raw 100%  ->  clean   0%
OpenAI     'Background checks'            raw 100%  ->  clean   0%
Databricks 'Benefits At Databricks'       raw 100%  ->  clean   0%
NVIDIA     'most desirable employers'     raw  57%  ->  clean   0%


## 3. Before and after, one posting per company (first 300 characters)

In [4]:
for company, g in df.groupby("company_name"):
    r = g.iloc[3]
    print(f"=== {company}: {r.job_title}")
    print("RAW:    ", cut(r.job_description))
    print("CLEAN:  ", cut(r.cleaned_description))
    print("NLP:    ", cut(r.nlp_text, 200), "\n")

=== Anthropic: Technical Program Manager, Safeguards (Infrastructure & Evals)
RAW:     About Anthropic Anthropic’s mission is to create reliable, interpretable, and steerable AI systems. We want AI to be safe and beneficial for our users and for society as a whole. Our team is a quickly growing group of committed researchers, engineers, policy experts, and business leaders working tog...
CLEAN:   About the Role Safeguards Engineering builds and operates the infrastructure that keeps Anthropic's AI systems safe in production — the classifiers, detection pipelines, evaluation platforms, and monitoring systems that sit between our models and the real world. That infrastructure needs to be not j...
NLP:     role safeguard engineering build operate infrastructure anthropic ai safe production classifier detection pipeline evaluation platform monitor sit model real world infrastructure need just correct rel... 

=== Databricks: Lakebase Associate Director, Healthcare & Life Sciences
RAW:     

## 4. Two outputs for two jobs

- `cleaned_description` keeps capitals and symbols, for **Task 4** skill matching ("AWS", "Go", "C++" need exact case/symbols).
- `nlp_text` is lowercased, tokenised, stopwords removed and lemmatised (models → model), for **TF-IDF and similarity** in Tasks 6 and 8.

The tokeniser keeps tech terms whole. The lemmatiser was wrecking short tech words, so it only runs on words of 5+ letters:

In [5]:
test = "Physical AI and ML on AWS: you'll build GPUs, C++ and Node.js apps with Linux, CI/CD and GPT-4."
print("Input: ", test)
print("Output:", pp.to_nlp_text(test))
import simplemma
print("\nWhy the 5-letter rule: the raw lemmatiser gives",
      {w: simplemma.lemmatize(w, lang="en") for w in ["ai", "ml", "aws"]})

Input:  Physical AI and ML on AWS: you'll build GPUs, C++ and Node.js apps with Linux, CI/CD and GPT-4.
Output: physical ai ml aws build gpu c++ node.js app linux ci/cd gpt-4

Why the 5-letter rule: the raw lemmatiser gives {'ai': 'be', 'ml': 'millileter', 'aws': 'aw'}


## 5. Non-English postings

Flagged (`is_english = False`), not deleted, so Task 4 can skip them. Two checks: mostly basic-Latin letters (catches Japanese) and few common French/German/Spanish words. The second check found 12 Anthropic Paris roles with an **English title but a French description**.

In [6]:
df[~df.is_english][["company_name", "job_title", "location"]].head(6)

,company_name,job_title,location
2643,Databricks,Enterprise Account Executive - Hunter (New Logo),"Tokyo, Japan"
2816,Databricks,デリバリーソリューションアーキテクト,"Tokyo, Japan"
3243,Anthropic,"Enterprise Account Executive, Public Sector","Madrid, Spain"
3250,Anthropic,"Enterprise Account Executive, FSI","Paris, France"
3259,Anthropic,Forward Deployed Engineer,"Paris, France"
3304,Anthropic,"\tEnterprise Account Executive, Automotive","Paris, France"


## 6. Seniority from job titles

Keyword rules, most specific first. "Product Manager" or "Account Manager" is a job type, not a management level, so those don't count as `manager`.

In [7]:
pd.crosstab(df.company_name, df.seniority_level)

seniority_level,director,executive,intern,junior,lead,manager,mid,principal,senior,staff
company_name,,,,,,,,,,
Anthropic,16,13,0,2,55,165,322,4,8,97
Databricks,39,4,13,7,19,151,305,10,242,136
NVIDIA,56,0,193,16,43,355,587,94,1393,18
OpenAI,49,8,0,6,101,133,497,4,14,6


## 7. Duplicate postings

The same role is often posted once per city (e.g. one Databricks role across 9 cities). These are flagged with `duplicate_count`, not removed: Task 5 decides whether hiring demand counts requisitions or unique roles.

In [8]:
(df[df.duplicate_count > 1].groupby(["company_name", "description_hash"])
   .agg(postings=("job_id", "size"), title=("job_title", "first"))
   .sort_values("postings", ascending=False).head(5))

postings  \
company_name description_hash             
Databricks   5226a070c267             9   
             f5b7c776d99e             9   
OpenAI       aef85d706c0e             7   
Databricks   6dbac7ab5372             7   
             d24b8a21c640             6   

                                                                           title  
company_name description_hash                                                     
Databricks   5226a070c267      Sr. Forward Deployed Engineer (FDE) - Communic...  
             f5b7c776d99e      Sr. Forward Deployed Engineer (FDE) - Financia...  
OpenAI       aef85d706c0e                       Applied AI Architect, Enterprise  
Databricks   6dbac7ab5372           Sr. Forward Deployed Engineer (FDE) - Retail  
             d24b8a21c640                          Sr. Forward Deployed Engineer

## 8. Result: comparable quality across companies

Before cleaning, median description length ranged from about 3,700 to 8,000 characters, mostly because of boilerplate. After cleaning the four companies sit much closer together, and no posting ended up empty.

In [9]:
pd.read_csv("../docs/preprocessing_stats.csv", index_col=0)

,postings,boilerplate_sentences_found,median_chars_before,median_chars_after,median_tokens,non_english,empty_after_cleaning,in_duplicate_groups,pct_text_removed
company_name,,,,,,,,,
Anthropic,682,34,8000.5,4064.5,358.5,12,0,20,49.2
Databricks,926,24,5368.0,3193.5,283.0,2,0,152,40.5
NVIDIA,2755,19,3742.0,2831.0,253.0,0,0,176,24.3
OpenAI,818,12,6545.5,3986.0,348.0,0,0,48,39.1
